# Séance 2 — Pandas : structures et exploration

**Durée pratique : 3 h 30** &nbsp;·&nbsp; Ateliers 2.1 à 2.3

## Ce que vous saurez faire à la fin

- charger une même source depuis trois formats et réconcilier les types obtenus ;
- sélectionner et filtrer sans déclencher de `SettingWithCopyWarning` ;
- produire automatiquement un rapport d'exploration sur un jeu de données inconnu.

Le jeu de travail est `ventes_brutes.csv` : environ 17 600 lignes de commandes,
volontairement dégradées. Vous allez le retrouver à chaque séance jusqu'à la fin du module.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  .gitkeep
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


---
## Atelier 2.1 — Lire une source, vraiment (50 min)

### Partie guidée : la lecture naïve et ce qu'elle cache

In [2]:
sales = pd.read_csv(RAW_DIR / 'ventes_brutes.csv')

print('Dimensions :', sales.shape)
sales.head()

Dimensions : (17609, 11)


,id_commande,date_commande,id_client,id_produit,id_magasin,quantite,prix_unitaire,remise_pct,canal,statut,ville_livraison
0,CMD0003112,2024/01/18 19:53,C02300,P0009,M05,3,160.66,5.0,telephone,retourne,Lille
1,CMD0000107,08/01/2024,C02791,P0028,M03,2,70.48,NaN,boutique,livre,Toulouse
2,CMD0010050,2023-04-15,990031,P0017,M04,3,691.12,0.0,boutique,livre,MARSEILLE
3,CMD0001979,06 Nov 2024,C01883,P0010,M04,4,570.25,0.0,WEB,retourne,Marseille
4,CMD0010513,2024-11-02,C00571,P0024,M02,2,1058.05,10.0,web,livre,Marseille


In [3]:
sales.dtypes

id_commande            str
date_commande          str
id_client              str
id_produit             str
id_magasin             str
quantite             int64
prix_unitaire          str
remise_pct         float64
canal                  str
statut                 str
ville_livraison        str
dtype: object

Regardez `prix_unitaire`. Le type est `object`, autrement dit du texte, alors qu'il s'agit
d'un montant. Cherchons pourquoi.

In [4]:
# Isoler les valeurs qui ne se convertissent pas en nombre
as_number = pd.to_numeric(sales['prix_unitaire'], errors='coerce')
unparsable = sales.loc[as_number.isna(), 'prix_unitaire']

print('Valeurs non convertibles :', len(unparsable))
print(unparsable.head(8).tolist())

Valeurs non convertibles : 1230
['129,12 EUR', '952,06 EUR', '1062,70 EUR', '591,16 EUR', '175,45 EUR', '524,21 EUR', '882,73 EUR', '1107,70 EUR']


Une partie des prix a été saisie avec une virgule décimale et un symbole monétaire.
Une lecture qui ignore ce détail produit une colonne texte, et tout calcul en aval échoue
silencieusement ou renvoie une erreur bien plus loin dans le pipeline.

**C'est la règle à retenir : ne jamais faire confiance à l'inférence de types.** Vérifiez
systématiquement `dtypes` après chaque lecture.

In [5]:
def parse_price(series):
    """Convertit une colonne de prix mixte (nombre ou texte '123,45 EUR') en float."""
    text = series.astype(str).str.replace(' EUR', '', regex=False)
    text = text.str.replace(',', '.', regex=False).str.strip()
    return pd.to_numeric(text, errors='coerce')


unit_price = parse_price(sales['prix_unitaire'])
print('Valeurs encore non convertibles :', unit_price.isna().sum())
print(unit_price.describe().round(2))

Valeurs encore non convertibles : 0
count    17609.00
mean       616.94
std        344.75
min         60.90
25%        278.66
50%        583.33
75%        926.79
max       1287.23
Name: prix_unitaire, dtype: float64


### Partie autonome

In [6]:
# Q1. Charger l'extrait des ventes depuis les trois formats disponibles,
#     puis comparer les dtypes obtenus pour la colonne 'date_commande'.

sample_xlsx = pd.read_excel(RAW_DIR / 'ventes_extrait.xlsx')
sample_json = pd.read_json(RAW_DIR / 'ventes_extrait.json')
# Pas d'extrait CSV fourni : l'extrait correspond aux premières lignes de ventes_brutes.csv
sample_csv = pd.read_csv(RAW_DIR / 'ventes_brutes.csv', nrows=len(sample_xlsx))

for label, frame in [('csv', sample_csv), ('xlsx', sample_xlsx), ('json', sample_json)]:
    print(f"{label:<6} lignes={len(frame):<6} dtype date={frame['date_commande'].dtype}")

csv    lignes=4000   dtype date=str
xlsx   lignes=4000   dtype date=str
json   lignes=4000   dtype date=str


**Question à traiter par écrit dans la cellule suivante.** Les trois formats donnent-ils
le même type pour `date_commande` ? Le même nombre de lignes ? Que se passerait-il si votre
pipeline acceptait indifféremment ces trois sources ?

*Votre réponse :*

**Même nombre de lignes ?** Oui : 4 000 dans les trois cas, parce que la version CSV est lue avec
`nrows=len(sample_xlsx)`. Ce n'est pas une propriété des formats : sans cette précaution, le CSV
complet en compte 17 609.

**Même type pour `date_commande` ?** Oui, `str` partout, mais pour une mauvaise raison : les dates sont écrites
dans cinq formats différents (`2023-01-31`, `15 Apr 2024`, `2024/02/06 16:05`…), si bien qu'aucun lecteur
ne les reconnaît comme des dates. Aucun des trois ne donne un vrai `datetime`.

**La vraie différence est ailleurs, sur `prix_unitaire`.** Le CSV ne stocke que du texte : la colonne entière
est `str`. Excel et JSON stockent des **valeurs typées** : la même colonne devient `object`, un mélange de
`float` (160.66), de `str` ('829,94 EUR') et même de `int` dans le fichier Excel. Le même fichier logique
donne donc trois colonnes différentes selon sa source.

**Conséquence pour un pipeline qui accepterait les trois sources.** Les traitements en aval se comporteraient
différemment selon l'origine du fichier : un `.str.replace` échoue sur les `float` de l'Excel, une
comparaison marche avec l'un et plante avec l'autre, une concaténation mélange les types. Il faut donc
**imposer le schéma à l'entrée** : lister les types attendus (`dtype=`), convertir explicitement les dates
(`pd.to_datetime` avec un format connu) et les prix (`parse_price`), puis vérifier `dtypes` avant d'aller plus loin.

In [7]:
# Q2. Relire ventes_brutes.csv en une seule instruction, en imposant :
#     - id_client, id_produit, id_magasin comme chaînes de caractères ;
#     - la chaîne vide et 'NC' comme valeurs manquantes ;
#     - seulement les colonnes id_commande, date_commande, id_produit, quantite,
#       prix_unitaire, statut.
#     Indice : paramètres dtype, na_values et usecols.

EXPECTED_COLUMNS = ['id_commande', 'date_commande', 'id_produit',
                    'quantite', 'prix_unitaire', 'statut']

sales_subset = pd.read_csv(
    RAW_DIR / 'ventes_brutes.csv',
    usecols=EXPECTED_COLUMNS,
    dtype={'id_client': object, 'id_produit': object, 'id_magasin': object},
    na_values=['', 'NC'],
)[EXPECTED_COLUMNS]  # TODO

assert list(sales_subset.columns) == EXPECTED_COLUMNS
assert sales_subset['id_produit'].dtype == object
print('OK —', sales_subset.shape)

OK — (17609, 6)


**Pourquoi cette réponse (Q2).**
- `usecols` : on ne charge que les six colonnes utiles, ce qui économise de la mémoire et du temps.
- `dtype={...: object}` : on **impose** le texte pour les identifiants au lieu de laisser pandas deviner.
  C'est essentiel ici : 347 identifiants clients ressemblent à des nombres (`990031`). Si pandas les
  lisait comme des entiers, un identifiant comme `00123` perdrait ses zéros et les jointures casseraient.
  pandas accepte qu'on type `id_client` même si la colonne n'est pas chargée : l'entrée est simplement ignorée.
- `na_values=['', 'NC']` : « NC » (non communiqué) est une convention métier pour « valeur absente ».
  La déclarer permet à `isna()` de la compter comme manquante au lieu d'une vraie modalité.
- `[EXPECTED_COLUMNS]` à la fin : `usecols` garde l'ordre **du fichier**, pas celui de la liste ; cette
  sélection remet les colonnes dans l'ordre attendu par l'`assert`.

---
## Atelier 2.2 — Sélection et filtrage (60 min)

### Partie guidée : `loc`, `iloc` et le piège de la copie

In [8]:
sales['prix_unitaire'] = parse_price(sales['prix_unitaire'])
sales['montant'] = sales['quantite'] * sales['prix_unitaire']

# loc travaille sur les ÉTIQUETTES (noms de colonnes, valeurs d'index)
print(sales.loc[0:2, ['id_commande', 'quantite', 'montant']])
print()
# iloc travaille sur les POSITIONS entières
print(sales.iloc[0:2, [0, 5, -1]])

  id_commande  quantite  montant
0  CMD0003112         3   481.98
1  CMD0000107         2   140.96
2  CMD0010050         3  2073.36

  id_commande  quantite  montant
0  CMD0003112         3   481.98
1  CMD0000107         2   140.96


Notez la différence sur les bornes : `loc[0:2]` renvoie **trois** lignes (borne incluse),
`iloc[0:2]` en renvoie **deux** (borne exclue, comme le slicing Python).

In [9]:
# Filtrage booléen : combiner des conditions avec & et |, chaque condition entre parenthèses
large_orders = sales[(sales['montant'] > 1000) & (sales['statut'] == 'livre')]
print('Commandes livrées de plus de 1000 EUR :', len(large_orders))

# query() est souvent plus lisible quand les conditions s'accumulent
same_result = sales.query("montant > 1000 and statut == 'livre'")
print('Même résultat :', len(same_result) == len(large_orders))

Commandes livrées de plus de 1000 EUR : 6039
Même résultat : True


In [10]:
# LE PIÈGE : modifier un sous-ensemble extrait par filtrage
cancelled = sales[sales['statut'] == 'annule']

# La ligne suivante déclenche un SettingWithCopyWarning : pandas ne sait pas si
# `cancelled` est une vue sur `sales` ou une copie indépendante.
cancelled['montant'] = 0

print('Montant dans sales pour les annulées :',
      sales.loc[sales['statut'] == 'annule', 'montant'].head(3).tolist())
print("-> la modification n'a PAS été propagée : le travail est perdu")

Montant dans sales pour les annulées : [1637.24, 1077.2, 4642.95]
-> la modification n'a PAS été propagée : le travail est perdu


**Les deux écritures correctes**, selon l'intention :

```python
# Intention A : modifier le DataFrame d'origine
sales.loc[sales['statut'] == 'annule', 'montant'] = 0

# Intention B : travailler sur une copie indépendante
cancelled = sales[sales['statut'] == 'annule'].copy()
cancelled['montant'] = 0
```

Le message d'avertissement est le symptôme d'une ambiguïté dans votre code, pas un bruit
à faire taire.

### Partie autonome

In [11]:
# Q3. Extraire les commandes qui remplissent TOUTES ces conditions :
#     - statut 'livre' ;
#     - quantite comprise entre 1 et 10 inclus ;
#     - montant strictement supérieur à la médiane des montants des commandes livrées ;
#     - canal contenant 'web', quelle que soit la casse.
#     Le résultat doit être une COPIE indépendante.

is_delivered = sales['statut'] == 'livre'
median_delivered = sales.loc[is_delivered, 'montant'].median()
channel = sales['canal'].str.strip().str.lower()

target_orders = sales[
    is_delivered
    & sales['quantite'].between(1, 10)
    & (sales['montant'] > median_delivered)
    & channel.str.contains('web', na=False)
].copy()  # TODO

assert isinstance(target_orders, pd.DataFrame)
assert target_orders['quantite'].between(1, 10).all()
assert (target_orders['statut'] == 'livre').all()
assert target_orders['canal'].str.strip().str.lower().eq('web').all()
print('OK —', len(target_orders), 'commandes retenues')

OK — 2895 commandes retenues


**Pourquoi cette réponse (Q3).**
- La médiane est calculée **uniquement sur les commandes livrées** (`sales.loc[is_delivered, 'montant']`),
  comme le demande l'énoncé, et non sur tout le jeu.
- `between(1, 10)` inclut les deux bornes par défaut : c'est exactement « entre 1 et 10 inclus ».
- Pour le canal, `str.strip().str.lower()` neutralise d'abord les espaces et la casse (`'  WEB '` → `'web'`),
  sinon on raterait les modalités mal saisies vues dans l'atelier 2.3. `na=False` évite qu'une valeur
  manquante produise `NaN` dans le masque, ce qui ferait échouer le filtrage.
- Les conditions sont combinées avec `&`, chacune entre parenthèses, car `&` est prioritaire sur `>` et `==`.
- `.copy()` rend le résultat **indépendant** de `sales`, ce qui évite le piège du `SettingWithCopyWarning`
  vu juste avant.

In [12]:
# Q4. Sans utiliser groupby, calculer le montant total des commandes livrées
#     pour chacun des trois canaux (après normalisation de la casse).
#     Un dictionnaire {canal: total} est attendu.
channel = sales['canal'].str.strip().str.lower()
is_delivered = sales['statut'] == 'livre'

revenue_by_channel = {
    c: sales.loc[is_delivered & (channel == c), 'montant'].sum()
    for c in channel.dropna().unique()
}  # TODO

assert set(revenue_by_channel) == {'web', 'boutique', 'telephone'}
print('OK')
for channel_name, total in sorted(revenue_by_channel.items(), key=lambda item: -item[1]):
    print(f'  {channel_name:<12} {total:>14,.2f} EUR'.replace(',', ' '))

OK
  web           15 011 930.71 EUR
  boutique       9 861 116.79 EUR
  telephone      7 049 647.82 EUR


**Pourquoi cette réponse (Q4).**
On normalise d'abord le canal (espaces et casse), sinon on obtiendrait 9 totaux au lieu de 3.
Ensuite, pour chaque canal distinct, un masque booléen sélectionne les lignes livrées de ce canal, et
`.loc[masque, 'montant'].sum()` en fait la somme. Une compréhension de dictionnaire construit directement
`{canal: total}` sans `groupby`. On boucle sur **3 canaux**, pas sur 17 609 lignes : le calcul reste vectorisé.
Dans l'affichage, la variable s'appelle `channel_name` pour ne pas écraser la Series `channel`.
C'est l'équivalent de `sales[is_delivered].groupby(channel)['montant'].sum()`, qui sera vu plus tard.

---
## Atelier 2.3 — Un rapport d'exploration réutilisable (100 min)

Face à un jeu de données inconnu, les mêmes questions reviennent toujours. Plutôt que de
les reposer à la main à chaque fois, vous allez écrire une fonction qui y répond.
**Cette fonction vous servira jusqu'à la fin du module, y compris sur votre projet.**

### Partie guidée : les briques

In [13]:
print('--- shape ---'); print(sales.shape)
print('\n--- info ---'); sales.info(memory_usage='deep')

--- shape ---
(17609, 12)

--- info ---
<class 'pandas.DataFrame'>
RangeIndex: 17609 entries, 0 to 17608
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_commande      17609 non-null  str    
 1   date_commande    17259 non-null  str    
 2   id_client        17609 non-null  str    
 3   id_produit       17609 non-null  str    
 4   id_magasin       17609 non-null  str    
 5   quantite         17609 non-null  int64  
 6   prix_unitaire    17609 non-null  float64
 7   remise_pct       15668 non-null  float64
 8   canal            17609 non-null  str    
 9   statut           17609 non-null  str    
 10  ville_livraison  17609 non-null  str    
 11  montant          17609 non-null  float64
dtypes: float64(3), int64(1), str(8)
memory usage: 2.5 MB


In [14]:
print('--- taux de valeurs manquantes ---')
missing_rate = (sales.isna().mean() * 100).round(2).sort_values(ascending=False)
print(missing_rate[missing_rate > 0])

print('\n--- cardinalité des colonnes texte ---')
for column in sales.select_dtypes(include=['object', 'str']).columns:
    print(f'  {column:<18} {sales[column].nunique():>6} modalités')

--- taux de valeurs manquantes ---
remise_pct       11.02
date_commande     1.99
dtype: float64

--- cardinalité des colonnes texte ---
  id_commande         17369 modalités
  date_commande        6321 modalités
  id_client            3052 modalités
  id_produit             40 modalités
  id_magasin              9 modalités
  canal                   9 modalités
  statut                  3 modalités
  ville_livraison        33 modalités


In [15]:
print('--- modalités de canal, telles quelles ---')
print(sales['canal'].value_counts())

--- modalités de canal, telles quelles ---
canal
web             7013
boutique        4684
telephone       2352
WEB             1060
BOUTIQUE         724
  web            650
  boutique       525
TELEPHONE        364
  telephone      237
Name: count, dtype: int64


Neuf modalités pour ce qui devrait en compter trois. La casse et les espaces parasites
créent de faux niveaux. Un `value_counts()` brut est précisément l'outil qui révèle ce
genre de problème : c'est pourquoi il doit figurer dans le rapport automatique.

### Partie autonome : la fonction `profile_dataframe`

In [16]:
def profile_dataframe(df, name='jeu de données', max_cardinality=25):
    """Affiche un rapport d'exploration standard.

    Doit produire, dans cet ordre :
    1. le nom, les dimensions et l'empreinte mémoire ;
    2. le nombre de lignes strictement dupliquées ;
    3. un tableau par colonne : type, nombre de valeurs manquantes,
    taux en %, nombre de valeurs distinctes ;
    4. les statistiques descriptives des colonnes numériques ;
    5. pour chaque colonne texte de cardinalité inférieure à
    max_cardinality, la répartition des modalités.

    Ne renvoie rien : la fonction affiche.
    """

    memory_mb = df.memory_usage(deep=True).sum() / 1024**2

    print(f'=== {name} ===')
    print(
        f'Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes'
        f' | mémoire : {memory_mb:.2f} Mo'
    )

    print(f'Lignes dupliquées : {df.duplicated().sum()}')

    summary = pd.DataFrame({
        'type': df.dtypes.astype(str),
        'manquants': df.isna().sum(),
        'taux_%': (df.isna().mean() * 100).round(2),
        'distinctes': df.nunique()
    })

    print('\n--- colonnes ---')
    print(summary.to_string())

    numeric = df.select_dtypes(include='number')

    if not numeric.empty:
        print('\n--- statistiques numériques ---')
        print(numeric.describe().T.round(2).to_string())

    for column in df.select_dtypes(include=['object', 'string']).columns:
        n_unique = df[column].nunique()

        if n_unique < max_cardinality:
            print(f'\n--- {column} ({n_unique} modalités) ---')
            print(df[column].value_counts(dropna=False).to_string())
    print()


profile_dataframe(sales, name='ventes_brutes')

=== ventes_brutes ===
Dimensions : 17609 lignes x 12 colonnes | mémoire : 2.52 Mo
Lignes dupliquées : 240

--- colonnes ---
                    type  manquants  taux_%  distinctes
id_commande          str          0    0.00       17369
date_commande        str        350    1.99        6321
id_client            str          0    0.00        3052
id_produit           str          0    0.00          40
id_magasin           str          0    0.00           9
quantite           int64          0    0.00          68
prix_unitaire    float64          0    0.00       15839
remise_pct       float64       1941   11.02           5
canal                str          0    0.00           9
statut               str          0    0.00           3
ville_livraison      str          0    0.00          33
montant          float64          0    0.00       16605

--- statistiques numériques ---


                 count     mean       std     min     25%      50%      75%         max
quantite       17609.0     4.38     41.13    -4.0    1.00     2.00     3.00     1199.00
prix_unitaire  17609.0   616.94    344.75    60.9  278.66   583.33   926.79     1287.23
remise_pct     15668.0     7.14      7.47     0.0    0.00     5.00    15.00       20.00
montant        17609.0  2706.76  29172.25 -4504.6  519.93  1029.32  1895.10  1103954.49

--- id_magasin (9 modalités) ---
id_magasin
M01    2013
M03    2004
M06    1976
M99    1967
M05    1938
M02    1935
M07    1930
M08    1925
M04    1921

--- canal (9 modalités) ---
canal
web             7013
boutique        4684
telephone       2352
WEB             1060
BOUTIQUE         724
  web            650
  boutique       525
TELEPHONE        364
  telephone      237

--- statut (3 modalités) ---
statut
livre       11788
annule       2964
retourne     2857



In [17]:
# Q5. Appliquer la fonction aux deux autres jeux de données et vérifier qu'elle
#     se comporte correctement sur des structures différentes.

customers = pd.read_csv(RAW_DIR / 'clients.csv')
sensors = pd.read_csv(RAW_DIR / 'capteurs.csv')

profile_dataframe(customers, name='clients')
profile_dataframe(sensors, name='capteurs')

=== clients ===
Dimensions : 3060 lignes x 5 colonnes | mémoire : 0.22 Mo


Lignes dupliquées : 0

--- colonnes ---
                     type  manquants  taux_%  distinctes
id_client             str          0    0.00        3060
date_inscription      str          0    0.00        1545
segment               str          0    0.00           3
ville                 str          0    0.00          32
age               float64        185    6.05          86



--- statistiques numériques ---
      count   mean    std   min   25%   50%   75%    max
age  2875.0  46.51  80.88 -16.0  31.0  40.0  49.0  999.0

--- segment (3 modalités) ---
segment
Particulier      1542
Professionnel     997
Association       521

=== capteurs ===
Dimensions : 38916 lignes x 5 colonnes | mémoire : 2.38 Mo


Lignes dupliquées : 300



--- colonnes ---
                  type  manquants  taux_%  distinctes
horodatage         str          0    0.00       12960
id_capteur         str          0    0.00           3
temperature_c  float64       1160    2.98        3190
humidite_pct   float64          0    0.00         660
pm25           float64          0    0.00         574

--- statistiques numériques ---
                 count   mean     std     min    25%    50%   75%     max
temperature_c  37756.0  12.01    7.17   -5.44   6.23  11.83  17.7   31.34
humidite_pct   38916.0  57.54  108.36 -999.00  60.78  68.50  76.0  100.00
pm25           38916.0  13.15    8.81    0.10   6.70  11.30  17.6   78.00



--- id_capteur (3 modalités) ---


id_capteur
CAP-C    13070
CAP-A    13069
CAP-B    12777



**Pourquoi cette réponse (Q5).**
La fonction ne suppose rien sur les colonnes : elle se base sur les **types** (`select_dtypes`) et sur
la **cardinalité**. Elle marche donc telle quelle sur `clients` (5 colonnes) comme sur `capteurs` (38 916 lignes,
aucune colonne commune avec les ventes). C'est ce qui la rend réutilisable. Ce qu'elle révèle :
- `clients` : des âges impossibles (min **-16**, max **999**, une valeur de remplissage) et 6 % d'âges manquants ;
- `capteurs` : **300 doublons stricts**, 3 % de températures manquantes, et une humidité minimale de **-999**,
  un code d'erreur du capteur et non une mesure.

Limite constatée : la colonne `ville` (32 modalités) dépasse `max_cardinality=25` et n'est pas détaillée,
alors qu'elle contient justement des problèmes de casse. Le seuil doit rester un paramètre à ajuster.

In [18]:
# Q6. Déplacer la fonction dans src/exploration.py, puis l'importer ici.
#     Le notebook doit rester lisible : le code réutilisable vit dans src/.

import sys
sys.path.insert(0, str(ROOT / 'src'))

from exploration import profile_dataframe

**Pourquoi cette réponse (Q6).**
Le code réutilisable vit dans `src/exploration.py` ; le notebook se contente de l'importer. Un seul endroit à
corriger quand la fonction évolue, et elle devient utilisable dans les autres séances et dans le projet.
`sys.path.insert(0, ...)` ajoute le dossier `src/` aux chemins où Python cherche les modules : sans cela,
`import exploration` échoue, car le notebook est lancé depuis la racine du projet.
Cet import remplace la version définie plus haut dans le notebook.

---
## Ce que le rapport révèle déjà

Rédigez ici, en cinq à dix lignes, la liste des anomalies que votre rapport a mises au jour
sur `ventes_brutes`. Ce texte est le point de départ de la séance 3 et le premier élément
de votre note méthodologique de projet.

*Vos observations :*

1. **Prix stockés en texte** : `prix_unitaire` est lu comme `str` ; 1 230 valeurs sont au format `'829,94 EUR'`
   (virgule décimale et devise). Il faut les convertir avant tout calcul.
2. **Dates hétérogènes et manquantes** : cinq formats coexistent (`2023-01-31`, `15 Apr 2024`, `2024/02/06 16:05`…)
   et 350 dates manquent (1,99 %).
3. **Remises manquantes** : 1 941 valeurs de `remise_pct` absentes (11,02 %). Il faudra savoir si l'absence
   signifie « pas de remise » ou « non saisie ».
4. **Doublons stricts** : 240 lignes strictement identiques, qui gonfleraient le chiffre d'affaires.
5. **Quantités aberrantes** : minimum **-4** (impossible) et maximum **1 199** (improbable) pour une médiane de 2.
   Le `montant` va donc de -4 504 € à plus de 1,1 M €.
6. **Casse et espaces instables** : 9 modalités de `canal` pour 3 réelles (`web`, `WEB`, `'  web '`…),
   33 modalités de `ville_livraison` pour une dizaine de villes (`Lyon`, `lyon`, `LYON`…).
7. **Identifiants clients hors format** : 347 ventes portent un `id_client` au format `990031` au lieu de `C0xxxx`.
   Ces identifiants existent bien dans `clients.csv`, mais ce sont des **doublons** d'autres clients (étudiés en
   séance 3) : l'historique d'achat de ces clients est éclaté sur deux identifiants.

---
## Livrable de la séance

- `src/exploration.py` contenant la fonction `profile_dataframe`, importable ;
- ce notebook exécuté, avec les six questions complétées ;
- la liste écrite des anomalies constatées.